In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from getpass import getpass

MYSQL_JAR = (
    r"D:\Big Data Programming Project\Final Assignment"
    r"\drivers\mysql-connector-j-9.7.0.jar"
)

spark = (
    SparkSession.builder
    .appName("SCNE MySQL Database Integration")
    .master("local[*]")
    .config("spark.jars", MYSQL_JAR)
    .getOrCreate()
)

print("Spark version:", spark.version)
print("MySQL JDBC driver added.")

Spark version: 3.5.8
MySQL JDBC driver added.


In [2]:
DATA_PATH = (
    r"D:\Big Data Programming Project\Final Assignment"
    r"\data\processed\spark\scne_model_features"
)

df = spark.read.parquet(DATA_PATH)

print("Final dataset rows:", df.count())
print("Final dataset columns:", len(df.columns))
print("Partitions:", df.rdd.getNumPartitions())

Final dataset rows: 308885
Final dataset columns: 16
Partitions: 12


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --------------------------------------------------
# 1. Routes table
# --------------------------------------------------
route_window = Window.orderBy("published_line_name")

routes = (
    df.select("published_line_name")
    .where(F.col("published_line_name").isNotNull())
    .distinct()
    .withColumn(
        "route_id",
        F.row_number().over(route_window).cast("long")
    )
    .select(
        "route_id",
        F.col("published_line_name").alias("route_name")
    )
)

# Used to connect route names with route IDs
route_lookup = routes.select(
    F.col("route_name").alias("published_line_name"),
    "route_id"
)

# --------------------------------------------------
# 2. Service dates table
# --------------------------------------------------
service_dates = (
    df.select(
        F.to_date("service_date").alias("service_date"),
        F.col("day_of_week").cast("int"),
        F.col("is_weekend").cast("int"),
        F.col("is_public_holiday").cast("int")
    )
    .dropDuplicates(["service_date"])
)

# --------------------------------------------------
# 3. Stops table
# --------------------------------------------------
stops = (
    df.select("stop_id")
    .where(F.col("stop_id").isNotNull())
    .distinct()
)

# --------------------------------------------------
# 4. Trips table
# --------------------------------------------------
trips = (
    df.select(
        "trip_id",
        F.to_date("service_date").alias("service_date"),
        "published_line_name",
        F.col("direction_id").cast("int")
    )
    .dropDuplicates(["trip_id", "service_date"])
    .join(
        route_lookup,
        on="published_line_name",
        how="inner"
    )
    .select(
        "trip_id",
        "service_date",
        F.col("route_id").cast("long"),
        "direction_id"
    )
)

# --------------------------------------------------
# 5. Journey observations table
# --------------------------------------------------
observation_window = Window.orderBy(
    "service_date",
    "trip_id",
    "stop_sequence",
    "stop_id"
)

journey_observations = (
    df.withColumn(
        "observation_id",
        F.row_number().over(observation_window).cast("long")
    )
    .select(
        "observation_id",
        "trip_id",
        F.to_date("service_date").alias("service_date"),
        "stop_id",
        F.col("stop_sequence").cast("int"),
        F.col("hour").cast("int"),
        F.col("minute").cast("int"),
        F.col("journey_progress").cast("double"),
        F.col("previous_stop_delay").cast("double"),
        F.col("rolling_previous_delay").cast("double"),
        F.col("has_previous_delay").cast("int"),
        F.col("delay_seconds").cast("double")
    )
)

# --------------------------------------------------
# Check the table sizes
# --------------------------------------------------
print("Routes:", routes.count())
print("Service dates:", service_dates.count())
print("Stops:", stops.count())
print("Trips:", trips.count())
print("Journey observations:", journey_observations.count())

Routes: 62
Service dates: 3
Stops: 4026
Trips: 8526
Journey observations: 308885


In [1]:
from getpass import getpass

MYSQL_HOST = "localhost"
MYSQL_PORT = "3306"
MYSQL_DATABASE = "scne_bus_delay"
MYSQL_USER = "root"

MYSQL_PASSWORD = getpass("Enter your MySQL password: ")

JDBC_URL = (
    f"jdbc:mysql://{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
    "?sslMode=DISABLED&allowPublicKeyRetrieval=true"
)

JDBC_PROPERTIES = {
    "user": MYSQL_USER,
    "password": MYSQL_PASSWORD,
    "driver": "com.mysql.cj.jdbc.Driver"
}

print("MySQL connection settings prepared.")

Enter your MySQL password:  ········


MySQL connection settings prepared.


In [5]:
try:
    test_mysql = (
        spark.read
        .format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", "routes")
        .option("user", MYSQL_USER)
        .option("password", MYSQL_PASSWORD)
        .option("driver", "com.mysql.cj.jdbc.Driver")
        .load()
    )

    print("MySQL connection successful.")
    print("Current routes in MySQL:", test_mysql.count())

except Exception as error:
    print("MySQL connection failed.")
    print(error)

MySQL connection successful.
Current routes in MySQL: 0


In [6]:
def write_to_mysql(dataframe, table_name, partitions=1):
    (
        dataframe.coalesce(partitions)
        .write
        .format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", table_name)
        .option("user", MYSQL_USER)
        .option("password", MYSQL_PASSWORD)
        .option("driver", "com.mysql.cj.jdbc.Driver")
        .option("batchsize", "5000")
        .mode("append")
        .save()
    )

    print(f"Written successfully: {table_name}")

In [7]:
write_to_mysql(routes, "routes")
write_to_mysql(service_dates, "service_dates")
write_to_mysql(stops, "stops")
write_to_mysql(trips, "trips", partitions=2)

print("The four smaller tables were written to MySQL.")

Written successfully: routes
Written successfully: service_dates
Written successfully: stops
Written successfully: trips
The four smaller tables were written to MySQL.


In [8]:
write_to_mysql(
    journey_observations,
    "journey_observations",
    partitions=4
)

print("Journey observations were written to MySQL.")

Written successfully: journey_observations
Journey observations were written to MySQL.


In [9]:
table_names = [
    "routes",
    "service_dates",
    "stops",
    "trips",
    "journey_observations"
]

for table_name in table_names:
    mysql_df = (
        spark.read
        .format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", table_name)
        .option("user", MYSQL_USER)
        .option("password", MYSQL_PASSWORD)
        .option("driver", "com.mysql.cj.jdbc.Driver")
        .load()
    )

    print(f"{table_name}: {mysql_df.count()} records")

routes: 62 records
service_dates: 3 records
stops: 4026 records
trips: 8526 records
journey_observations: 308885 records


In [10]:
mysql_observations = (
    spark.read
    .format("jdbc")
    .option("url", JDBC_URL)
    .option("dbtable", "journey_observations")
    .option("user", MYSQL_USER)
    .option("password", MYSQL_PASSWORD)
    .option("driver", "com.mysql.cj.jdbc.Driver")
    .load()
)

mysql_observations.show(5, truncate=False)

+--------------+------------------------------------------+------------+------------+-------------+----+------+-------------------+-------------------+----------------------+------------------+-------------+
|observation_id|trip_id                                   |service_date|stop_id     |stop_sequence|hour|minute|journey_progress   |previous_stop_delay|rolling_previous_delay|has_previous_delay|delay_seconds|
+--------------+------------------------------------------+------------+------------+-------------+----+------+-------------------+-------------------+----------------------+------------------+-------------+
|1             |VJ00281d66b067d9e9e4b472236e1e40548cea038c|2025-12-26  |410000007134|0            |22  |39    |0.0                |0.0                |0.0                   |0                 |4.0          |
|2             |VJ00281d66b067d9e9e4b472236e1e40548cea038c|2025-12-26  |410000014578|2            |22  |41    |0.06060606060606061|4.0                |4.0              

In [2]:
import mysql.connector

print("MySQL Connector/Python is available.")

MySQL Connector/Python is available.


In [3]:
import mysql.connector

connection = None
cursor = None

try:
    connection = mysql.connector.connect(
        host=MYSQL_HOST,
        port=int(MYSQL_PORT),
        database=MYSQL_DATABASE,
        user=MYSQL_USER,
        password=MYSQL_PASSWORD
    )

    cursor = connection.cursor()

    selected_route = "X78"
    selected_date = "2025-12-28"

    query = """
        SELECT
            r.route_name,
            j.service_date,
            COUNT(*) AS observation_count,
            ROUND(AVG(j.delay_seconds), 2) AS average_delay_seconds
        FROM journey_observations AS j
        INNER JOIN trips AS t
            ON j.trip_id = t.trip_id
            AND j.service_date = t.service_date
        INNER JOIN routes AS r
            ON t.route_id = r.route_id
        WHERE r.route_name = %s
          AND j.service_date = %s
        GROUP BY r.route_name, j.service_date
    """

    cursor.execute(
        query,
        (selected_route, selected_date)
    )

    results = cursor.fetchall()

    print("Parameterized query result:")

    for row in results:
        print(row)

except mysql.connector.Error as error:
    print("MySQL error:", error)

finally:
    if cursor is not None:
        cursor.close()

    if connection is not None and connection.is_connected():
        connection.close()

Parameterized query result:
('X78', datetime.date(2025, 12, 28), 493, 564.52)
